In [5]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, XPreformatted
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from google.colab import files
import re
import pandas as pd
import numpy as np

# --- Normalization Mapping ---
COLLEGE_ALIAS_MAP = {
    'IIMLUCKNOW': 'IIML',
    'IIMBANGALORE': 'IIMB',
    'IIMAHMEDABAD': 'IIMA',
    'IIMCALCUTTA': 'IIMC',
    'IIMINDORE': 'IIMI',
    'IIMKOZHIKODE': 'IIMK',
    'IIMSHILLONG': 'IIMS',
    'IIMMUMBAI': 'IIMM',
    'IIMBPGPBA': 'IIMB',
    'MDIGURGAON': 'MDI',
    'MDIG': 'MDI',
    'XLRIBM': 'XLRI',
    'XLRIHRM': 'XLRI',
    'XLRIHR': 'XLRI',
    'XLRI(BM)': 'XLRI',
    'XLRI(HRM)': 'XLRI',
    'IITKGPVGSOM': 'IITKGP',
    'IIMUDAIPUR': 'IIMU'
}

def normalize_college_name(name):
    if pd.isna(name): return name
    clean = re.sub(r'[^A-Z0-9]', '', str(name).upper())
    for alias, canonical in COLLEGE_ALIAS_MAP.items():
        if clean == alias: return canonical
    return clean

# --- Parsing Functions ---

def load_and_parse_transcripts(file_path):
    with open(file_path, 'r') as file:
        file_content = file.read()
    parsed_data = []
    file_content_stripped = file_content.strip()
    transcript_blocks = re.findall(r'((?:#Real Interview Transcript(?: #THEOMI)?|#THEOMI)\s*[\s\S]*?(?=(?:#Real Interview Transcript|#THEOMI)|\Z))', file_content_stripped, re.DOTALL)
    transcript_blocks = [block.strip() for block in transcript_blocks if block.strip()]
    for block in transcript_blocks:
        transcript_info = {}
        single_line_patterns = {
            'Transcript No': r'Transcript No: (.*?)(?=\n|$)',
            'College': r'College: (.*?)(?=\n|$)',
            'Date': r'Date: (.*?)(?=\n|$)',
            'UG': r'UG: (.*?)(?=\n|$)',
            'PG': r'PG: (.*?)(?=\n|$)',
            'Work Ex (months)': r'Work Ex \(months\): (.*?)(?=\n|$)',
            'Certifications': r'Certifications: (.*?)(?=\n|$)'
        }
        for field, pattern in single_line_patterns.items():
            match = re.search(pattern, block)
            transcript_info[field] = match.group(1).strip() if match else 'NA'
        bg_match = re.search(r'Background Notes:\n([\s\S]*?)(?=\n\nExperience Notes:|\n\nQ&A:|\Z)', block, re.DOTALL)
        transcript_info['Background Notes'] = bg_match.group(1).strip() if bg_match else 'NA'
        exp_match = re.search(r'Experience Notes:\n([\s\S]*?)(?=\n\nQ&A:|\Z)', block, re.DOTALL)
        transcript_info['Experience Notes'] = exp_match.group(1).strip() if exp_match else 'NA'
        qa_match = re.search(r'Q&A:\n([\s\S]*)', block, re.DOTALL)
        transcript_info['Q&A'] = qa_match.group(1).strip() if qa_match else 'NA'
        parsed_data.append(transcript_info)
    df = pd.DataFrame(parsed_data)
    df['Work Ex (months)'] = pd.to_numeric(df['Work Ex (months)'], errors='coerce')
    return df

def clean_transcript_dataframe(df):
    df = df.replace(['NA', 'Na'], np.nan, regex=True)
    df['na_count'] = df.isnull().sum(axis=1)
    df = df[df['na_count'] < 8].drop(columns=['na_count'])
    df['College'] = df['College'].apply(normalize_college_name)
    return df

def load_and_parse_wat_data(file_path):
    with open(file_path, 'r') as file:
        wat_file_content = file.read()
    parsed_wat_data = []
    wat_blocks = re.findall(r'(^College[ :-][\s\S]*?)(?=(?:^College[ :-]|\Z))', wat_file_content, re.DOTALL | re.MULTILINE)
    for block in wat_blocks:
        wat_info = {}
        patterns = {
            'College': r'^College[ :-]\s*(.*?)$',
            'Date': r'^Date[ :-]\s*(.*?)$',
            'Slot': r'^Slot[ :-]\s*(.*?)$',
            'Time': r'^Time[ :-]\s*(.*?)$',
            'Topic': r'^(?:Topic|WAT topic|GD topic)[ :-]\s*([\s\S]*)$'
        }
        for field, pattern in patterns.items():
            match = re.search(pattern, block, re.MULTILINE | (re.IGNORECASE if field == 'Topic' else 0))
            wat_info[field] = match.group(1).strip() if match else np.nan
        parsed_wat_data.append(wat_info)
    return pd.DataFrame(parsed_wat_data)

def clean_wat_dataframe(wat_df):
    # FILTER: Remove entries where Topic is missing or empty
    wat_df = wat_df.dropna(subset=['Topic'])
    wat_df = wat_df[wat_df['Topic'].str.strip() != '']

    def parse_wat_date(date_str):
        if pd.isna(date_str) or not isinstance(date_str, str): return pd.NaT
        clean_date_str = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date_str.strip()).replace('-', '').strip()
        try: return pd.to_datetime(f"{clean_date_str} 2026", format='%d %b %Y', errors='coerce')
        except: return pd.NaT
    wat_df['Date'] = wat_df['Date'].apply(parse_wat_date)
    wat_df['College_Normalized'] = wat_df['College'].apply(normalize_college_name)
    return wat_df

# --- Sorting Logic ---
iim_preferred_order = ['IIMA', 'IIMB', 'IIMC', 'IIML', 'IIMK', 'IIMI', 'IIMS', 'IIMM']
non_iim_preferred_order = ['SPJIMR', 'XLRI', 'MDI', 'IIFT']

def get_custom_college_sort_key(college_name):
    if pd.isna(college_name): return (4, 0, '')
    college_name_canonical = college_name.strip().upper()
    if college_name_canonical in iim_preferred_order:
        return (0, iim_preferred_order.index(college_name_canonical), college_name_canonical)
    if 'IIM' in college_name_canonical:
        return (1, 0, college_name_canonical)
    if college_name_canonical in non_iim_preferred_order:
        return (2, non_iim_preferred_order.index(college_name_canonical), college_name_canonical)
    return (3, 0, college_name_canonical)

def add_line_breaks(text, char_limit=110):
    if not isinstance(text, str): return text
    lines = text.split('\n')
    processed_lines = []
    for line in lines:
        if not line.strip(): processed_lines.append(line); continue
        words = line.split()
        current_wrapped_line = []
        current_length = 0
        for word in words:
            if current_length + len(word) + (1 if current_wrapped_line else 0) > char_limit and current_length > 0:
                processed_lines.append(' '.join(current_wrapped_line))
                current_wrapped_line = [word]
                current_length = len(word)
            else:
                current_wrapped_line.append(word)
                current_length += len(word) + (1 if current_wrapped_line else 0)
        if current_wrapped_line: processed_lines.append(' '.join(current_wrapped_line))
    return '\n'.join(processed_lines)

def format_transcript_entry(row):
    def get_val(v): return str(v).strip() if pd.notna(v) and str(v).strip() != '' else 'Not Available'
    return f"\n-------------------------------------------------------\nTranscript No: {get_val(row['Transcript No'])}\nCollege: {get_val(row['College'])}\nDate: {get_val(row['Date'])}\nUG: {get_val(row['UG'])}\nPG: {get_val(row['PG'])}\nWork Ex (months): {get_val(row['Work Ex (months)'])}\nCertifications: {get_val(row['Certifications'])}\n\nBackground Notes:\n{add_line_breaks(get_val(row['Background Notes']))}\n\nExperience Notes:\n{add_line_breaks(get_val(row['Experience Notes']))}\n\nQ&A:\n{add_line_breaks(get_val(row['Q&A']))}\n------------------------------------------------------- "

def format_wat_entry(row):
    def get_val(v):
        if pd.isna(v) or str(v).strip() == '': return 'Not Available'
        return v.strftime('%d %b %Y') if isinstance(v, pd.Timestamp) else str(v).strip()
    return f"\n-------------------------------------------------------\nCollege: {get_val(row['College'])}\nDate: {get_val(row['Date'])}\nSlot: {get_val(row['Slot'])}\nTime: {get_val(row['Time'])}\nTopic: {add_line_breaks(get_val(row['Topic']))}\n------------------------------------------------------- "

def page_template_with_watermark(canvas_obj, doc_obj):
    canvas_obj.saveState()
    canvas_obj.setFont('Helvetica-Bold', 80)
    canvas_obj.setFillColorRGB(0.95, 0.95, 0.95)
    width, height = letter
    canvas_obj.translate(width/2, height/2)
    canvas_obj.rotate(45)
    text_width = canvas_obj.stringWidth("#THEOMI", 'Helvetica-Bold', 80)
    canvas_obj.drawString(-text_width/2, -40, "#THEOMI")
    canvas_obj.restoreState()

def generate_interview_pdf(transcript_file_path, wat_file_path, output_filename):
    df_raw = load_and_parse_transcripts(transcript_file_path)
    df = clean_transcript_dataframe(df_raw.copy())
    college_counts = df['College'].value_counts()
    colleges_to_keep = college_counts[college_counts > 2].index
    df_filtered = df[df['College'].isin(colleges_to_keep)].copy()
    df_filtered['College_Sort_Key'] = df_filtered['College'].apply(get_custom_college_sort_key)
    df_filtered['Date'] = pd.to_datetime(df_filtered['Date'], errors='coerce')
    df_filtered = df_filtered.sort_values(by=['College_Sort_Key', 'Date'], ascending=[True, True]).drop(columns=['College_Sort_Key']).reset_index(drop=True)
    formatted_transcripts_filtered = [format_transcript_entry(row) for _, row in df_filtered.iterrows()]

    wat_df_raw = load_and_parse_wat_data(wat_file_path)
    wat_df = clean_wat_dataframe(wat_df_raw.copy())
    wat_df['College_Sort_Key'] = wat_df['College_Normalized'].apply(get_custom_college_sort_key)
    wat_df_sorted = wat_df.sort_values(by=['College_Sort_Key', 'Date'])
    wat_topics_grouped = {}
    for _, row in wat_df_sorted.iterrows():
        c = row['College_Normalized']
        if c not in wat_topics_grouped: wat_topics_grouped[c] = []
        wat_topics_grouped[c].append(format_wat_entry(row))

    # Calculate counts for Table of Contents
    wat_counts = {college: len(topics) for college, topics in wat_topics_grouped.items()}
    transcript_counts = df_filtered['College'].value_counts().to_dict()

    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name='ChapterTitle', parent=styles['h1'], fontSize=24, spaceAfter=14))
    styles.add(ParagraphStyle(name='TranscriptBody', parent=styles['Normal'], fontName='Helvetica', fontSize=10, leading=12, spaceAfter=6))

    doc = SimpleDocTemplate(output_filename, pagesize=letter, leftMargin=0.5*inch, rightMargin=0.5*inch, topMargin=0.75*inch, bottomMargin=0.75*inch)
    story = []
    story.append(Paragraph("MBA/IIM Interview Transcripts + WAT Topics Compilation", styles['ChapterTitle']))
    story.append(Spacer(1, 0.2*inch))
    story.append(Paragraph("Detailed analysis for Coaching", styles['h2']))
    story.append(Spacer(1, 0.2*inch))
    story.append(Paragraph("Compiled by - Ayush Agarwal (https://www.linkedin.com/in/ayush-agarwal-261041215/)", styles['h2']))
    story.append(PageBreak())

    story.append(Paragraph("Table of Contents", styles['h2']))
    story.append(Spacer(1, 0.2*inch))
    story.append(Paragraph("WAT Topics", styles['Normal']))
    for college in sorted(wat_topics_grouped.keys(), key=get_custom_college_sort_key):
        count = wat_counts.get(college, 0)
        story.append(Paragraph(f"  - {college} ({count})", styles['Normal']))
    story.append(Spacer(1, 0.1*inch))
    story.append(Paragraph("Interview Transcripts", styles['Normal']))
    unique_colleges = df_filtered['College'].unique().tolist()
    for college in sorted(unique_colleges, key=get_custom_college_sort_key):
        count = transcript_counts.get(college, 0)
        story.append(Paragraph(f"  - {college} ({count})", styles['Normal']))
    story.append(PageBreak())

    story.append(Paragraph("WAT Topics", styles['ChapterTitle']))
    for college in sorted(wat_topics_grouped.keys(), key=get_custom_college_sort_key):
        story.append(Spacer(1, 0.3*inch))
        story.append(Paragraph(college, styles['h3']))
        for block in wat_topics_grouped[college]:
            story.append(XPreformatted(block, styles['TranscriptBody']))
    story.append(PageBreak())

    story.append(Paragraph("Interview Transcripts", styles['ChapterTitle']))
    current_col = None
    for i, row in df_filtered.iterrows():
        if row['College'] != current_col:
            if current_col is not None: story.append(PageBreak())
            story.append(Paragraph(row['College'], styles['h3']))
            current_col = row['College']
        story.append(XPreformatted(formatted_transcripts_filtered[i], styles['TranscriptBody']))

    doc.build(story, onFirstPage=page_template_with_watermark, onLaterPages=page_template_with_watermark)
    print(f"PDF '{output_filename}' generated.")
    files.download(output_filename)

transcript_file_path='/content/transcripts_dataset.txt'
wat_file_path='/content/WAT_dataset.txt'
output_filename='final_interviews_wat_compilation.pdf'
generate_interview_pdf(transcript_file_path, wat_file_path, output_filename)

PDF 'final_interviews_wat_compilation.pdf' generated.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>